# UniFD 本地部署 + 模型配置 + 图片读取(文件夹/链接) + CSV导出 全流程

这个 Notebook 适用于 **UniversalFakeDetect (UniFD)** 代码库本地推理。

你可以：
1. 配置环境与模型参数；
2. 从**本地文件夹**批量读取图片；
3. 从**URL 列表**下载并读取图片；
4. 运行 UniFD 推理，得到真假概率与标签；
5. 导出 `csv` 结果文件。


## 0) 使用前说明

- 建议在仓库根目录执行本 Notebook（当前代码默认相对路径）。
- 默认模型：`CLIP:ViT-L/14` + `pretrained_weights/fc_weights.pth`。
- 输出标签约定：
  - `0` = real
  - `1` = fake


In [ ]:
# 如果你是全新环境，先安装依赖（已安装可跳过）
# !pip install -U pip
# !pip install torch torchvision packaging pillow pandas requests tqdm


In [ ]:
import os
import io
import csv
import json
import time
import random
from pathlib import Path
from typing import List, Dict, Optional

import torch
import pandas as pd
import requests
from PIL import Image
from tqdm.auto import tqdm
import torchvision.transforms as transforms

from models import get_model


In [ ]:
# ====== 可调参数区 ======
CONFIG = {
    # UniFD 模型结构（仓库里常用是 CLIP:ViT-L/14）
    "arch": "CLIP:ViT-L/14",

    # 线性分类头权重（仓库自带）
    "ckpt": "pretrained_weights/fc_weights.pth",

    # 推理阈值：sigmoid(logit) >= threshold 判为 fake
    "threshold": 0.5,

    # 批处理大小（显存不足就调小）
    "batch_size": 16,

    # 设备：自动优先 CUDA
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # 支持图片后缀
    "image_exts": [".jpg", ".jpeg", ".png", ".bmp", ".webp"],

    # 结果目录
    "output_dir": "inference_outputs",

    # URL 下载目录（临时文件）
    "url_cache_dir": "inference_outputs/url_images",

    # CSV 输出名
    "csv_name": "unifd_predictions.csv",
}

Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["url_cache_dir"]).mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))


In [ ]:
# 与 validate.py 一致的归一化参数
MEAN = {
    "imagenet": [0.485, 0.456, 0.406],
    "clip": [0.48145466, 0.4578275, 0.40821073],
}
STD = {
    "imagenet": [0.229, 0.224, 0.225],
    "clip": [0.26862954, 0.26130258, 0.27577711],
}


def build_transform(arch: str):
    stat_from = "imagenet" if arch.lower().startswith("imagenet") else "clip"
    return transforms.Compose([
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN[stat_from], std=STD[stat_from]),
    ])


def load_unifd_model(arch: str, ckpt_path: str, device: str):
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"未找到 checkpoint: {ckpt_path}")

    model = get_model(arch)
    state_dict = torch.load(ckpt_path, map_location="cpu")
    model.fc.load_state_dict(state_dict)
    model.eval().to(device)
    return model


In [ ]:
# 1) 模型加载
transform = build_transform(CONFIG["arch"])
model = load_unifd_model(CONFIG["arch"], CONFIG["ckpt"], CONFIG["device"])
print(f"Model loaded on {CONFIG['device']}: {CONFIG['arch']}")


In [ ]:
# 2) 数据读取工具：本地文件夹

def list_images_from_folder(folder: str, exts: List[str]) -> List[str]:
    folder_path = Path(folder)
    if not folder_path.exists():
        raise FileNotFoundError(f"文件夹不存在: {folder}")

    files = []
    for p in folder_path.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            files.append(str(p))

    files.sort()
    return files


In [ ]:
# 3) 数据读取工具：URL 列表 -> 下载到本地缓存目录

def sanitize_name(name: str) -> str:
    safe = "".join(c if c.isalnum() or c in "._-" else "_" for c in name)
    return safe[:120] if len(safe) > 120 else safe


def download_image(url: str, save_dir: str, timeout: int = 15) -> Optional[str]:
    try:
        resp = requests.get(url, timeout=timeout)
        resp.raise_for_status()
        content_type = resp.headers.get("Content-Type", "")

        # 尝试推断后缀
        ext = ".jpg"
        if "png" in content_type:
            ext = ".png"
        elif "webp" in content_type:
            ext = ".webp"
        elif "bmp" in content_type:
            ext = ".bmp"

        name = sanitize_name(url.split("/")[-1].split("?")[0])
        if not name:
            name = f"img_{int(time.time() * 1000)}"

        if not Path(name).suffix:
            name = name + ext

        save_path = Path(save_dir) / name
        with open(save_path, "wb") as f:
            f.write(resp.content)
        return str(save_path)
    except Exception:
        return None


def download_images_from_urls(urls: List[str], save_dir: str) -> pd.DataFrame:
    rows = []
    for url in tqdm(urls, desc="Downloading"):
        local_path = download_image(url, save_dir)
        rows.append({
            "source_type": "url",
            "source": url,
            "local_path": local_path,
            "download_ok": local_path is not None,
        })
    return pd.DataFrame(rows)


In [ ]:
# 4) 推理函数

def predict_image_paths(
    model,
    image_paths: List[str],
    transform,
    device: str,
    batch_size: int = 16,
    threshold: float = 0.5,
) -> pd.DataFrame:
    rows = []

    # 先过滤可打开的图片
    valid_items = []
    for p in image_paths:
        try:
            img = Image.open(p).convert("RGB")
            tensor = transform(img)
            valid_items.append((p, tensor))
        except Exception as e:
            rows.append({
                "image_path": p,
                "logit": None,
                "fake_prob": None,
                "pred_label": None,
                "pred_name": None,
                "error": str(e),
            })

    # 批量推理
    with torch.no_grad():
        for i in tqdm(range(0, len(valid_items), batch_size), desc="Inferencing"):
            batch = valid_items[i:i+batch_size]
            paths = [x[0] for x in batch]
            tensors = torch.stack([x[1] for x in batch], dim=0).to(device)

            logits = model(tensors).flatten()
            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).long()

            for p, l, pr, pd_ in zip(paths, logits.cpu(), probs.cpu(), preds.cpu()):
                pd_int = int(pd_.item())
                rows.append({
                    "image_path": p,
                    "logit": float(l.item()),
                    "fake_prob": float(pr.item()),
                    "pred_label": pd_int,
                    "pred_name": "fake" if pd_int == 1 else "real",
                    "error": "",
                })

    return pd.DataFrame(rows)


## 5) 方式A：从本地文件夹读取并预测


In [ ]:
# 把这里改成你的本地图片目录
LOCAL_IMAGE_DIR = "./demo_images"

local_image_paths = list_images_from_folder(LOCAL_IMAGE_DIR, CONFIG["image_exts"])
print(f"Found {len(local_image_paths)} images from folder: {LOCAL_IMAGE_DIR}")

df_local = predict_image_paths(
    model=model,
    image_paths=local_image_paths,
    transform=transform,
    device=CONFIG["device"],
    batch_size=CONFIG["batch_size"],
    threshold=CONFIG["threshold"],
)

print(df_local.head())


## 6) 方式B：从 URL 列表读取并预测

你可以直接写一个 URL 列表，或从文本文件加载。


In [ ]:
# 示例：直接写 URL（替换成你自己的）
URLS = [
    # "https://example.com/a.jpg",
    # "https://example.com/b.png",
]

# 或者从 txt 读取（每行一个 URL）
# with open("url_list.txt", "r", encoding="utf-8") as f:
#     URLS = [line.strip() for line in f if line.strip()]

if len(URLS) == 0:
    print("URLS 为空，跳过 URL 推理。")
    df_url_pred = pd.DataFrame()
    df_url_download = pd.DataFrame()
else:
    df_url_download = download_images_from_urls(URLS, CONFIG["url_cache_dir"])
    ok_paths = df_url_download.loc[df_url_download["download_ok"], "local_path"].tolist()

    df_url_pred = predict_image_paths(
        model=model,
        image_paths=ok_paths,
        transform=transform,
        device=CONFIG["device"],
        batch_size=CONFIG["batch_size"],
        threshold=CONFIG["threshold"],
    )

    # 把 source(url) 映射回预测表
    path_to_src = {
        r["local_path"]: r["source"]
        for _, r in df_url_download[df_url_download["download_ok"]].iterrows()
    }
    df_url_pred["source_url"] = df_url_pred["image_path"].map(path_to_src)

print("Download summary:")
if 'df_url_download' in globals() and not df_url_download.empty:
    display(df_url_download.head())


## 7) 合并结果并导出 CSV


In [ ]:
frames = []

if 'df_local' in globals() and not df_local.empty:
    _tmp = df_local.copy()
    _tmp["source_type"] = "folder"
    _tmp["source"] = _tmp["image_path"]
    frames.append(_tmp)

if 'df_url_pred' in globals() and not df_url_pred.empty:
    _tmp = df_url_pred.copy()
    _tmp["source_type"] = "url"
    if "source_url" in _tmp.columns:
        _tmp["source"] = _tmp["source_url"]
    else:
        _tmp["source"] = _tmp["image_path"]
    frames.append(_tmp)

if len(frames) == 0:
    print("没有可导出的预测结果（请先执行文件夹或 URL 推理）。")
else:
    df_all = pd.concat(frames, ignore_index=True)
    out_csv = Path(CONFIG["output_dir"]) / CONFIG["csv_name"]
    df_all.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"CSV 已导出: {out_csv}")
    print(f"总条数: {len(df_all)}")
    display(df_all.head(20))


## 8) 输出字段说明

- `image_path`: 本地图片路径（URL 图片为下载后的本地缓存路径）
- `logit`: 分类头输出 logit
- `fake_prob`: `sigmoid(logit)`，越接近 1 越可能是 fake
- `pred_label`: 预测标签（0=real, 1=fake）
- `pred_name`: 标签文本（real/fake）
- `source_type`: 来源类型（folder/url）
- `source`: 原始来源（本地路径或原 URL）
- `error`: 读取失败时的报错
